# Phase 1: Monaco 2025 Race Data Collection

## Objective

The goal of this notebook is to collect Formula 1 race data for the 2025 Monaco Grand Prix using the FastF1 Python package.

This notebook loads the race session, extracts lap-level data, weather data, and race result data, and saves the raw datasets for later cleaning, exploratory analysis, tyre degradation analysis, and machine learning modeling.

## Why Monaco 2025?

Monaco is one of the most strategy-sensitive circuits on the Formula 1 calendar. Because overtaking is difficult, lap time, tyre life, pit stop timing, traffic, and track position can strongly influence the race outcome. This makes Monaco a useful starting point for building a machine learning project focused on race pace and strategy analysis.

In [2]:
pip install fastf1

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.12.2
    Uninstalling typing_extensions-4.12.2:
      Successfully uninstalled typing_extensions-4.12.2m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
  Attempting uninstall: msgpack;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
    Found existing installation: msgpack 1.0.35;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
    Uninstalling msgpack-1.0.3:;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
      Successfully uninstalled msgpack-1.0.38;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
  Attempting uninstall: attrs;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/9 [typing-extensions]
    Found existing installation: attrs 24.3.0;5;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
# Import required libraries

import fastf1
import pandas as pd
from pathlib import Path

In [4]:
# Creating project folders

RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
CACHE_DIR = Path("../data/cache")

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("Folders are ready.")

Folders are ready.


In [5]:
# Enable FastF1 cache

fastf1.Cache.enable_cache(str(CACHE_DIR))

print("FastF1 cache enabled.")

FastF1 cache enabled.


## Load the Monaco 2025 Race Session

FastF1 allows us to load race sessions by year, race name, and session type.

For this project:

- Year: 2025
- Race: Monaco
- Session: Race

In [7]:
# Define race/session details

YEAR = 2025
RACE_NAME = "Monaco"
SESSION_TYPE = "R"   # R = Race

session = fastf1.get_session(YEAR, RACE_NAME, SESSION_TYPE)
session.load()

print(f"Loaded session: {YEAR} {RACE_NAME} Grand Prix - Race")

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cac

Loaded session: 2025 Monaco Grand Prix - Race


In [8]:
# Extract main datasets

laps = session.laps.copy()
weather = session.weather_data.copy()
results = session.results.copy()

print("Laps shape:", laps.shape)
print("Weather shape:", weather.shape)
print("Results shape:", results.shape)

Laps shape: (1425, 31)
Weather shape: (160, 8)
Results shape: (20, 22)


In [9]:
# Preview lap data

laps.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:57:36.090000,VER,1,0 days 00:01:27.020000,1.0,1.0,NaT,NaT,NaT,0 days 00:00:38.394000,...,True,Red Bull Racing,0 days 00:56:08.809000,2025-05-25 13:03:09.195,12,4.0,False,,False,False
1,0 days 00:59:22.408000,VER,1,0 days 00:01:46.318000,2.0,1.0,NaT,NaT,0 days 00:00:34.949000,0 days 00:00:45.145000,...,True,Red Bull Racing,0 days 00:57:36.090000,2025-05-25 13:04:36.476,16,4.0,False,,False,False
2,0 days 01:01:11.223000,VER,1,0 days 00:01:48.815000,3.0,1.0,NaT,NaT,0 days 00:00:32.110000,0 days 00:00:47.430000,...,True,Red Bull Racing,0 days 00:59:22.408000,2025-05-25 13:06:22.794,6,4.0,False,,False,False
3,0 days 01:02:51.513000,VER,1,0 days 00:01:40.290000,4.0,1.0,NaT,NaT,0 days 00:00:31.006000,0 days 00:00:47.889000,...,True,Red Bull Racing,0 days 01:01:11.223000,2025-05-25 13:08:11.609,671,4.0,False,,False,False
4,0 days 01:04:12.313000,VER,1,0 days 00:01:20.800000,5.0,1.0,NaT,NaT,0 days 00:00:21.744000,0 days 00:00:37.485000,...,True,Red Bull Racing,0 days 01:02:51.513000,2025-05-25 13:09:51.899,1,4.0,False,,False,True


In [10]:
# Preview available lap columns

laps.columns.tolist()

['Time',
 'Driver',
 'DriverNumber',
 'LapTime',
 'LapNumber',
 'Stint',
 'PitOutTime',
 'PitInTime',
 'Sector1Time',
 'Sector2Time',
 'Sector3Time',
 'Sector1SessionTime',
 'Sector2SessionTime',
 'Sector3SessionTime',
 'SpeedI1',
 'SpeedI2',
 'SpeedFL',
 'SpeedST',
 'IsPersonalBest',
 'Compound',
 'TyreLife',
 'FreshTyre',
 'Team',
 'LapStartTime',
 'LapStartDate',
 'TrackStatus',
 'Position',
 'Deleted',
 'DeletedReason',
 'FastF1Generated',
 'IsAccurate']

In [11]:
# Preview weather data

weather.head()

,Time,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed
0,0 days 00:00:18.586000,21.5,55.0,1017.3,False,42.6,298,1.0
1,0 days 00:01:18.584000,21.6,54.0,1017.3,False,43.6,176,0.7
2,0 days 00:02:18.615000,21.6,54.0,1017.3,False,43.5,180,2.2
3,0 days 00:03:18.615000,21.4,55.0,1017.2,False,43.3,0,1.2
4,0 days 00:04:18.632000,21.3,55.0,1017.1,False,43.2,341,0.4


In [12]:
# Preview results data

results.head()

,DriverNumber,BroadcastName,Abbreviation,DriverId,TeamName,TeamColor,TeamId,FirstName,LastName,FullName,...,Position,ClassifiedPosition,GridPosition,Q1,Q2,Q3,Time,Status,Points,Laps
4,4,L NORRIS,NOR,norris,McLaren,F47600,mclaren,Lando,Norris,Lando Norris,...,1.0,1,1.0,NaT,NaT,NaT,0 days 01:40:33.843000,Finished,25.0,78.0
16,16,C LECLERC,LEC,leclerc,Ferrari,ED1131,ferrari,Charles,Leclerc,Charles Leclerc,...,2.0,2,2.0,NaT,NaT,NaT,0 days 00:00:03.131000,Finished,18.0,78.0
81,81,O PIASTRI,PIA,piastri,McLaren,F47600,mclaren,Oscar,Piastri,Oscar Piastri,...,3.0,3,3.0,NaT,NaT,NaT,0 days 00:00:03.658000,Finished,15.0,78.0
1,1,M VERSTAPPEN,VER,max_verstappen,Red Bull Racing,4781D7,red_bull,Max,Verstappen,Max Verstappen,...,4.0,4,4.0,NaT,NaT,NaT,0 days 00:00:20.572000,Finished,12.0,78.0
44,44,L HAMILTON,HAM,hamilton,Ferrari,ED1131,ferrari,Lewis,Hamilton,Lewis Hamilton,...,5.0,5,7.0,NaT,NaT,NaT,0 days 00:00:51.387000,Finished,10.0,78.0


## Save Raw Data

The raw FastF1 data will be saved into the `data/raw` folder. These files will not be heavily cleaned in this notebook. Cleaning and transformation will happen in the next phase.

In [13]:
# Save raw datasets

laps_path = RAW_DATA_DIR / "2025_monaco_race_laps.csv"
weather_path = RAW_DATA_DIR / "2025_monaco_weather.csv"
results_path = RAW_DATA_DIR / "2025_monaco_results.csv"

laps.to_csv(laps_path, index=False)
weather.to_csv(weather_path, index=False)
results.to_csv(results_path, index=False)

print("Saved files:")
print(laps_path)
print(weather_path)
print(results_path)

Saved files:
../data/raw/2025_monaco_race_laps.csv
../data/raw/2025_monaco_weather.csv
../data/raw/2025_monaco_results.csv


In [14]:
# Quick check of important columns

important_columns = [
    "Driver",
    "Team",
    "LapNumber",
    "LapTime",
    "Compound",
    "TyreLife",
    "Stint",
    "TrackStatus",
    "Position",
    "PitInTime",
    "PitOutTime"
]

available_columns = [col for col in important_columns if col in laps.columns]
missing_columns = [col for col in important_columns if col not in laps.columns]

print("Available important columns:")
print(available_columns)

print("\nMissing columns:")
print(missing_columns)

Available important columns:
['Driver', 'Team', 'LapNumber', 'LapTime', 'Compound', 'TyreLife', 'Stint', 'TrackStatus', 'Position', 'PitInTime', 'PitOutTime']

Missing columns:
[]


## Phase 1 Summary

In this notebook, we loaded the 2025 Monaco Grand Prix race session using FastF1 and saved three raw datasets:

- Lap-level race data
- Weather data
- Race result data

These files will be used in the next phase to clean lap times, remove invalid laps, identify pit laps, and prepare the dataset for exploratory analysis and tyre degradation analysis.